# Emotion-Conditioned German VITS — MELD-ST Dataset
### CMU 11492/11692/18495 — Term Project (TTS Component)

**Pipeline:**
```
English Speech → [Whisper ASR + Emotion Recognizer] → emotion_label
                                                            ↓
               → [Translation] → German text               ↓
                                                            ↓
                         → [Emotion-Conditioned VITS] → German speech
```

**Dataset:** ku-nlp/MELD-ST ENG_DEU — German speech translations of MELD with emotion labels

**Model:** Coqui `tts_models/de/thorsten/vits` (native German) + `emb_g: Embedding(7, 256)`

**Training:** Freeze all weights except `emb_g` — preserves German speech quality, only learns emotion conditioning

**Emotions:** neutral, surprise, fear, sadness, joy, disgust, anger

## 0) Check GPU

In [1]:
import torch, os

assert torch.cuda.is_available(), 'No GPU! Go to Runtime → Change runtime type → A100 GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}')

DEVICE = 'cuda'
from pathlib import Path
os.makedirs('tts_result', exist_ok=True)

GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB
PyTorch: 2.10.0+cu128


## 1) Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
EXP_DIR  = Path('/content/drive/MyDrive/vits_meld_coqui_st')
PREP_DIR = Path('/content/data_prep')
EXP_DIR.mkdir(parents=True, exist_ok=True)
PREP_DIR.mkdir(parents=True, exist_ok=True)
print(f'EXP_DIR:  {EXP_DIR}')
print(f'Existing: {list(EXP_DIR.iterdir()) if any(EXP_DIR.iterdir()) else "(empty — fresh start)"}')

Mounted at /content/drive
EXP_DIR:  /content/drive/MyDrive/vits_meld_coqui_st
Existing: [PosixPath('/content/drive/MyDrive/vits_meld_coqui_st/1epoch.pth'), PosixPath('/content/drive/MyDrive/vits_meld_coqui_st/2epoch.pth'), PosixPath('/content/drive/MyDrive/vits_meld_coqui_st/3epoch.pth'), PosixPath('/content/drive/MyDrive/vits_meld_coqui_st/best_full.pth'), PosixPath('/content/drive/MyDrive/vits_meld_coqui_st/4epoch.pth'), PosixPath('/content/drive/MyDrive/vits_meld_coqui_st/5epoch.pth'), PosixPath('/content/drive/MyDrive/vits_meld_coqui_st/6epoch.pth'), PosixPath('/content/drive/MyDrive/vits_meld_coqui_st/7epoch.pth'), PosixPath('/content/drive/MyDrive/vits_meld_coqui_st/8epoch.pth'), PosixPath('/content/drive/MyDrive/vits_meld_coqui_st/9epoch.pth'), PosixPath('/content/drive/MyDrive/vits_meld_coqui_st/10epoch.pth'), PosixPath('/content/drive/MyDrive/vits_meld_coqui_st/checkpoint.pth'), PosixPath('/content/drive/MyDrive/vits_meld_coqui_st/best.pth')]


## 2) Install Dependencies

In [3]:
!pip install coqui-tts --quiet
!pip install librosa soundfile tqdm pandas numpy datasets huggingface_hub --quiet
!apt-get install -y ffmpeg --quiet

print('Installation complete.')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.8/862.8 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.3/997.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 648.4/648.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 2.9 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
Installation complete.


## 3) Session Setup (run after every Colab restart)

In [4]:
import torch, os
from pathlib import Path

DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
EXP_DIR  = Path('/content/drive/MyDrive/vits_meld_coqui_st')
PREP_DIR = Path('/content/data_prep')
PREP_DIR.mkdir(parents=True, exist_ok=True)

EMOTIONS      = ['neutral', 'surprise', 'fear', 'sadness', 'joy', 'disgust', 'anger']
EMOTION_TO_ID = {emo: idx for idx, emo in enumerate(EMOTIONS)}
NUM_CLASSES   = len(EMOTIONS)  # 7
EMB_DIM       = 256

print(f'Device: {DEVICE} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')
print(f'Emotions: {EMOTION_TO_ID}')

Device: cuda | GPU: NVIDIA A100-SXM4-40GB
Emotions: {'neutral': 0, 'surprise': 1, 'fear': 2, 'sadness': 3, 'joy': 4, 'disgust': 5, 'anger': 6}


## 4) Load MELD-ST Dataset from HuggingFace

In [8]:
from huggingface_hub import login
from datasets import load_dataset

# Login with HuggingFace token

print('Loading MELD-ST ENG_DEU dataset...')
ds = load_dataset('ku-nlp/MELD-ST', 'ENG_DEU')
print(ds)
print('\nColumn names:', ds['train'].column_names)
print('\nSample entry:')
print(ds['train'][0])

Loading MELD-ST ENG_DEU dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

deu_train.csv: 0.00B [00:00, ?B/s]

deu_test.csv: 0.00B [00:00, ?B/s]

deu_valid.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/9314 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1164 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/1164 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dia_id', 'utt_id', 'emotion', 'sentiment', 'English', 'German', 'season', 'episode', 'speaker', 'English_begin_time', 'English_end_time', 'German_begin_time', 'German_end_time'],
        num_rows: 9314
    })
    test: Dataset({
        features: ['id', 'dia_id', 'utt_id', 'emotion', 'sentiment', 'English', 'German', 'season', 'episode', 'speaker', 'English_begin_time', 'English_end_time', 'German_begin_time', 'German_end_time'],
        num_rows: 1164
    })
    valid: Dataset({
        features: ['id', 'dia_id', 'utt_id', 'emotion', 'sentiment', 'English', 'German', 'season', 'episode', 'speaker', 'English_begin_time', 'English_end_time', 'German_begin_time', 'German_end_time'],
        num_rows: 1164
    })
})

Column names: ['id', 'dia_id', 'utt_id', 'emotion', 'sentiment', 'English', 'German', 'season', 'episode', 'speaker', 'English_begin_time', 'English_end_time', 'German_begin_time', 'German_end_time']

Sample entry:


In [9]:
import zipfile, os
from pathlib import Path

# Unzip to /content/
ZIP_PATH = '/content/ENG_DEU.zip'  # update path if different

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall('/content/')

# Find the DEU folder
for root, dirs, files in os.walk('/content'):
    if 'DEU' in dirs or root.endswith('DEU'):
        print(f'Found DEU at: {root}')
        break

# List what's there
!ls /content/ENG_DEU/DEU/
!ls /content/ENG_DEU/DEU/train/ | head -5

FileNotFoundError: [Errno 2] No such file or directory: '/content/ENG_DEU.zip'

## 5) Inspect Dataset — Verify German Text, Audio, and Emotion Labels

In [ ]:
import random
import numpy as np
from IPython.display import Audio, display

print('=== 5 random samples: German text + emotion + audio ===\n')
indices = random.sample(range(len(ds['train'])), 5)

for idx in indices:
    sample = ds['train'][idx]
    print(f'Index:   {idx}')

    # Print all available fields so we can see the schema
    for k, v in sample.items():
        if k != 'audio':  # skip audio bytes for printing
            print(f'{k}: {v}')

    # Play audio if present
    if 'audio' in sample and sample['audio'] is not None:
        audio_data = sample['audio']
        if isinstance(audio_data, dict):
            wav = np.array(audio_data['array'], dtype=np.float32)
            sr  = audio_data['sampling_rate']
        else:
            wav = np.array(audio_data, dtype=np.float32)
            sr  = 22050
        print(f'Audio:   {len(wav)/sr:.1f}s @ {sr}Hz')
        display(Audio(wav, rate=sr))
    print('─' * 60)

=== 5 random samples: German text + emotion + audio ===

Index:   6396
id: 6396
dia_id: 643
utt_id: 6
emotion: neutral
sentiment: neutral
English: Hey look, Phoebe. I, uh, I laid out the states geographically...
German: Hör zu. Ich habe die Bundesstaaten geografisch...
season: 7
episode: 8
speaker: Ross
English_begin_time: 00:14:50,681
English_end_time: 00:14:54,268
German_begin_time: 00:14:50,681
German_end_time: 00:14:54,268
────────────────────────────────────────────────────────────
Index:   7317
id: 7317
dia_id: 773
utt_id: 2
emotion: neutral
sentiment: neutral
English: Men are here.
German: (MIT TIEFER STIMME) Männer hier sein!
season: 1
episode: 24
speaker: Chandler
English_begin_time: 00:02:36,448
English_end_time: 00:02:38,367
German_begin_time: 00:02:36,448
German_end_time: 00:02:37,824
────────────────────────────────────────────────────────────
Index:   668
id: 668
dia_id: 48
utt_id: 1
emotion: neutral
sentiment: neutral
English: Okay, okay, but Rachel’s gonna be here too, 

## 6) Build Data Prep from MELD-ST

Extracts German text, emotion labels, and saves audio to disk.

In [ ]:
import soundfile as sf
import numpy as np
from pathlib import Path
from tqdm import tqdm
from collections import Counter

AUDIO_BASE = Path('/content/ENG_DEU/DEU')
PREP_DIR.mkdir(exist_ok=True)

def build_split(hf_split, audio_split_name, prep_split_name):
    split_data   = ds[hf_split]
    audio_folder = AUDIO_BASE / audio_split_name
    out_dir      = PREP_DIR / prep_split_name
    out_dir.mkdir(exist_ok=True)

    if not audio_folder.exists():
        print(f'Audio folder not found: {audio_folder}')
        return 0

    entries, skipped = [], 0
    for sample in tqdm(split_data, desc=f'Processing {prep_split_name}'):
        try:
            sample_id = sample['id']
            text      = str(sample.get('German', '')).strip()
            emotion   = str(sample.get('emotion', 'neutral')).strip().lower()

            if not text:
                skipped += 1
                continue
            if emotion not in EMOTION_TO_ID:
                emotion = 'neutral'

            wav_path = audio_folder / f'{audio_split_name}_{sample_id}.wav'
            if not wav_path.exists():
                skipped += 1
                continue

            utt_id = f'meldst_{prep_split_name}_{sample_id:05d}'
            entries.append({
                'utt_id':  utt_id,
                'wav':     str(wav_path.resolve()),
                'text':    text,
                'cond_id': str(EMOTION_TO_ID[emotion]),
                'emotion': emotion,
            })
        except Exception:
            skipped += 1
            continue

    with open(out_dir / 'wav.scp', 'w') as fw, \
         open(out_dir / 'text',    'w') as ft, \
         open(out_dir / 'utt2spk', 'w') as fu:
        for e in entries:
            fw.write(f"{e['utt_id']} {e['wav']}\n")
            ft.write(f"{e['utt_id']} {e['text']}\n")
            fu.write(f"{e['utt_id']} {e['cond_id']}\n")

    print(f'{prep_split_name}: {len(entries)} entries, {skipped} skipped')
    emo_counts = Counter(e['emotion'] for e in entries)
    for emo, count in sorted(emo_counts.items()):
        print(f'  {emo:12s}: {count}')
    return len(entries)

print('Available HF splits:', list(ds.keys()))

for hf_name, audio_name, prep_name in [
    ('train', 'train', 'train'),
    ('valid', 'dev',   'val'),
    ('test',  'test',  'test'),
]:
    if hf_name in ds:
        build_split(hf_name, audio_name, prep_name)

print('\nData prep complete.')

Available HF splits: ['train', 'test', 'valid']


Processing train: 100%|██████████| 9314/9314 [00:01<00:00, 5238.85it/s]


train: 9314 entries, 0 skipped
  anger       : 1096
  disgust     : 261
  fear        : 232
  joy         : 1571
  neutral     : 4402
  sadness     : 656
  surprise    : 1096


Processing val: 100%|██████████| 1164/1164 [00:00<00:00, 5477.45it/s]


val: 1164 entries, 0 skipped
  anger       : 127
  disgust     : 25
  fear        : 31
  joy         : 202
  neutral     : 550
  sadness     : 99
  surprise    : 130


Processing test: 100%|██████████| 1164/1164 [00:00<00:00, 5219.46it/s]

test: 1164 entries, 0 skipped
  anger       : 102
  disgust     : 39
  fear        : 32
  joy         : 218
  neutral     : 550
  sadness     : 92
  surprise    : 131

Data prep complete.


## 7) Verify Data — Text, Audio, Emotion

In [ ]:
import soundfile as sf
from IPython.display import Audio, display
import random

with open(PREP_DIR / 'train' / 'utt2spk') as f:
    emo_lookup  = {l.split()[0]: int(l.split()[1]) for l in f.readlines()}
with open(PREP_DIR / 'train' / 'text') as f:
    text_lookup = {l.split()[0]: l.split(' ', 1)[1].strip() for l in f.readlines()}
with open(PREP_DIR / 'train' / 'wav.scp') as f:
    scp_lines = f.readlines()

print('=== 5 random samples: German text + emotion + audio ===\n')
for line in random.sample(scp_lines, 5):
    utt_id, wav_path = line.strip().split(' ', 1)
    text    = text_lookup.get(utt_id, '?')
    emo_id  = emo_lookup.get(utt_id, -1)
    emo_str = EMOTIONS[emo_id] if 0 <= emo_id < len(EMOTIONS) else '?'
    audio, sr = sf.read(wav_path)
    print(f'ID:      {utt_id}')
    print(f'Text:    {text}')
    print(f'Emotion: {emo_str} (ID={emo_id})')
    print(f'Audio:   {len(audio)/sr:.1f}s @ {sr}Hz')
    display(Audio(audio, rate=sr))
    print('─' * 60)

=== 5 random samples: German text + emotion + audio ===

ID:      meldst_train_01871
Text:    Du solltest Schriftstellerin werden. Du musst mir erzählen, wie der Roman weitergeht.
Emotion: joy (ID=4)
Audio:   8.8s @ 16000Hz


────────────────────────────────────────────────────────────
ID:      meldst_train_05240
Text:    Wie lange wird das dauern?
Emotion: neutral (ID=0)
Audio:   1.3s @ 16000Hz


────────────────────────────────────────────────────────────
ID:      meldst_train_05247
Text:    Weil du nicht genug gearbeitet hast.
Emotion: surprise (ID=1)
Audio:   2.6s @ 16000Hz


────────────────────────────────────────────────────────────
ID:      meldst_train_03711
Text:    Die Monogamie ist bekanntlich ein 2-schneidiges Modell. Vom anthropologischen... (ALLE SCHNARCHEN)
Emotion: neutral (ID=0)
Audio:   5.8s @ 16000Hz


────────────────────────────────────────────────────────────
ID:      meldst_train_01982
Text:    Warte, bis du auch sicher, dass du das gut durchdacht hast?
Emotion: neutral (ID=0)
Audio:   3.1s @ 16000Hz


────────────────────────────────────────────────────────────


## 8) Save Data to Drive (run once)

In [ ]:
import shutil
from pathlib import Path

drive_data = Path('/content/drive/MyDrive/meld_st_data')
drive_data.mkdir(exist_ok=True)

print('Saving data_prep to Drive (fast)...')
shutil.copytree('/content/data_prep', str(drive_data / 'data_prep'), dirs_exist_ok=True)
print('data_prep saved.')

# Audio is ~1-2GB — uncomment if you want to save it
# print('Saving audio to Drive (~10-15 min)...')
# shutil.copytree('/content/meld_st_audio', str(drive_data / 'meld_st_audio'), dirs_exist_ok=True)
# print('Audio saved.')

Saving data_prep to Drive (fast)...
data_prep saved.


## 9) Restore Data from Drive (run after Colab reset instead of cells 4-8)

In [7]:
import shutil
from pathlib import Path

drive_data = Path('/content/drive/MyDrive/meld_st_data')
shutil.copytree(str(drive_data / 'data_prep'), '/content/data_prep', dirs_exist_ok=True)
print('data_prep restored.')

# If you saved audio to Drive:
shutil.copytree(str(drive_data / 'meld_st_audio'), '/content/meld_st_audio', dirs_exist_ok=True)
print('Audio restored.')
# Otherwise re-run cells 4-6 to re-download and re-extract audio

data_prep restored.


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/meld_st_data/meld_st_audio'

## 10) Load Coqui German VITS

In [ ]:
!pip install coqui-tts[languages] --quiet
!pip install gruut[de] --quiet

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.3/85.3 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 52.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 12.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 134.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 109.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 83.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 147.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.2/33.2 MB 9.6 MB/s eta 0:00:00
  Preparing metadata (setu

In [ ]:
from TTS.api import TTS
import torch

print('Loading Coqui German VITS...')
tts_base = TTS(model_name='tts_models/de/thorsten/vits', progress_bar=True)
tts_base = tts_base.to(DEVICE)

SR = tts_base.synthesizer.output_sample_rate
print(f'Model loaded | SR: {SR} Hz')

Loading Coqui German VITS...
Model loaded | SR: 22050 Hz


## 11) Baseline Evaluation — BEFORE Finetuning

Establishes baseline: same German text, no emotion conditioning, all outputs identical.

In [ ]:
import soundfile as sf
import numpy as np
import librosa
import matplotlib.pyplot as plt
from IPython.display import Audio, display

synthesis_text = 'Ich kann nicht glauben, dass das gerade passiert.'
print(f'Baseline German TTS — no emotion conditioning')
print(f'Text: {synthesis_text}\n')
print('All outputs will be IDENTICAL since there is no emotion conditioning yet.\n')

# Listen to baseline
wav = tts_base.tts(text=synthesis_text)
wav_np = np.array(wav, dtype=np.float32)
sf.write('tts_result/baseline_german.wav', wav_np, SR)
print('Baseline audio:')
display(Audio(wav_np, rate=SR))

# Baseline F0
f0 = librosa.yin(wav_np, fmin=60, fmax=400, sr=SR)
f0_voiced = f0[(f0 > 60) & (f0 < 400)]
baseline_f0 = float(np.mean(f0_voiced)) if len(f0_voiced) > 0 else 0.0
print(f'\nBaseline F0: {baseline_f0:.1f} Hz')
print('After finetuning, each emotion will produce a different F0.')

# Baseline F0 bar chart — all bars should be identical
f0_baseline_all = {emo: baseline_f0 for emo in EMOTIONS}
colors = ['gray', 'purple', 'blue', 'steelblue', 'gold', 'green', 'red']
plt.figure(figsize=(9, 4))
plt.bar(f0_baseline_all.keys(), f0_baseline_all.values(), color=colors)
plt.ylabel('Mean F0 (Hz)')
plt.title('F0 Mean by Emotion — Baseline (no emotion conditioning)')
plt.ylim(0, 300)
plt.tight_layout()
plt.savefig('tts_result/f0_baseline.png', dpi=100)
plt.show()
print('Saved: tts_result/f0_baseline.png')

Baseline German TTS — no emotion conditioning
Text: Ich kann nicht glauben, dass das gerade passiert.

All outputs will be IDENTICAL since there is no emotion conditioning yet.



Baseline audio:



Baseline F0: 130.5 Hz
After finetuning, each emotion will produce a different F0.
Saved: tts_result/f0_baseline.png


## 12) Add Emotion Embedding

In [ ]:
import torch
import torch.nn as nn

vits_model = tts_base.synthesizer.tts_model
vits_model = vits_model.to(DEVICE)

print('Model type:', type(vits_model).__name__)
print('Has emb_g:', hasattr(vits_model, 'emb_g'))

if not hasattr(vits_model, 'emb_g') or vits_model.emb_g is None:
    vits_model.emb_g = nn.Embedding(NUM_CLASSES, EMB_DIM).to(DEVICE)
    nn.init.normal_(vits_model.emb_g.weight, mean=0.0, std=0.01)
    print(f'Added emb_g: Embedding({NUM_CLASSES}, {EMB_DIM})')
else:
    print(f'emb_g already exists: {vits_model.emb_g}')

print(f'Total params: {sum(p.numel() for p in vits_model.parameters()):,}')

Model type: Vits
Has emb_g: False
Added emb_g: Embedding(7, 256)
Total params: 83,060,972


## 13) Build Dataset

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import soundfile as sf
import numpy as np
from pathlib import Path
from collections import Counter

class MELDSTDataset(Dataset):
    def __init__(self, split, prep_dir, max_duration_sec=10.0, sr=22050):
        self.sr = sr
        self.max_samples = int(max_duration_sec * sr)
        self.entries = []

        scp   = open(prep_dir / split / 'wav.scp').readlines()
        texts = {l.split()[0]: l.split(' ', 1)[1].strip()
                 for l in open(prep_dir / split / 'text').readlines()}
        sids  = {l.split()[0]: int(l.split()[1])
                 for l in open(prep_dir / split / 'utt2spk').readlines()}

        for line in scp:
            utt_id, wav_path = line.strip().split(' ', 1)
            if utt_id in texts and utt_id in sids:
                self.entries.append({
                    'utt_id':   utt_id,
                    'wav_path': wav_path,
                    'text':     texts[utt_id],
                    'emotion':  sids[utt_id],
                })

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        e = self.entries[idx]
        audio, sr = sf.read(e['wav_path'], dtype='float32')
        if len(audio) > self.max_samples:
            audio = audio[:self.max_samples]
        return {
            'audio':   torch.FloatTensor(audio),
            'text':    e['text'],
            'emotion': torch.LongTensor([e['emotion']]),
            'utt_id':  e['utt_id'],
        }

def collate_fn(batch):
    max_len = max(b['audio'].shape[0] for b in batch)
    audio_padded = torch.zeros(len(batch), max_len)
    lengths = torch.LongTensor([b['audio'].shape[0] for b in batch])
    for i, b in enumerate(batch):
        audio_padded[i, :b['audio'].shape[0]] = b['audio']
    emotions = torch.cat([b['emotion'] for b in batch])
    texts = [b['text'] for b in batch]
    return audio_padded, lengths, emotions, texts

train_dataset = MELDSTDataset('train', PREP_DIR)
val_dataset   = MELDSTDataset('val',   PREP_DIR)

print(f'Train: {len(train_dataset)} utterances')
print(f'Val:   {len(val_dataset)} utterances')
print('\nEmotion distribution (train):')
emo_counts = Counter(e['emotion'] for e in train_dataset.entries)
for eid, count in sorted(emo_counts.items()):
    print(f'  {EMOTIONS[eid]:12s} (ID={eid}): {count}')

Train: 9314 utterances
Val:   1164 utterances

Emotion distribution (train):
  neutral      (ID=0): 4402
  surprise     (ID=1): 1096
  fear         (ID=2): 232
  sadness      (ID=3): 656
  joy          (ID=4): 1571
  disgust      (ID=5): 261
  anger        (ID=6): 1096


## 14) Training Setup — Freeze All Except emb_g

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

MAX_EPOCHS  = 50
BATCH_SIZE  = 8
LR          = 2e-4
LOG_EVERY   = 50
SAVE_EVERY  = 1

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_fn, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn, num_workers=2
)

# Freeze everything except emb_g
for name, param in vits_model.named_parameters():
    param.requires_grad = False
vits_model.emb_g.weight.requires_grad = True

trainable = sum(p.numel() for p in vits_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in vits_model.parameters())
print(f'Trainable: {trainable:,} / {total:,} params ({100*trainable/total:.4f}%)')
print(f'Only training: emb_g Embedding({NUM_CLASSES}, {EMB_DIM})')

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, vits_model.parameters()),
    lr=LR, betas=(0.8, 0.99), eps=1e-9
)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.999875)

CKPT_PATH   = EXP_DIR / 'checkpoint.pth'
start_epoch = 1

if CKPT_PATH.exists():
    print(f'Resuming from checkpoint: {CKPT_PATH}')
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
    vits_model.emb_g.weight.data = ckpt['emb_g_weight'].to(DEVICE)
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    start_epoch = ckpt['epoch'] + 1
    print(f'Resumed from epoch {ckpt["epoch"]} | loss: {ckpt.get("loss", "?"):.4f}')
else:
    print('No checkpoint — starting fresh from epoch 1')

print(f'Training epochs {start_epoch} → {MAX_EPOCHS} | {len(train_loader)} batches/epoch')

Trainable: 1,792 / 83,060,972 params (0.0022%)
Only training: emb_g Embedding(7, 256)
No checkpoint — starting fresh from epoch 1
Training epochs 1 → 50 | 1165 batches/epoch


In [ ]:
import inspect
print(inspect.signature(vits_model.forward))
print()
print('Model has these attributes related to g:')
for name in dir(vits_model):
    if 'emb' in name.lower() or 'global' in name.lower() or 'cond' in name.lower():
        print(f'  {name}')

(x: torch.Tensor, x_lengths: torch.Tensor, y: torch.Tensor, y_lengths: torch.Tensor, waveform: torch.Tensor, aux_input: dict[str, typing.Any] = {'d_vectors': None, 'speaker_ids': None, 'language_ids': None}) -> dict

Model has these attributes related to g:
  _init_speaker_embedding
  _named_members
  _set_cond_input
  emb_g
  embedded_language_dim
  embedded_speaker_dim


## 15) Training Loop

In [ ]:
# Enable speaker embedding path so emb_g is actually used
vits_model.args.use_speaker_embedding = True
vits_model.args.num_speakers = NUM_CLASSES
vits_model.num_speakers = NUM_CLASSES
vits_model.embedded_speaker_dim = EMB_DIM  # 256

# Re-freeze everything except emb_g (in case flipping flags unfroze stuff)
for name, param in vits_model.named_parameters():
    param.requires_grad = False
vits_model.emb_g.weight.requires_grad = True

trainable = sum(p.numel() for p in vits_model.parameters() if p.requires_grad)
print(f'Trainable params: {trainable:,}')
print(f'use_speaker_embedding: {vits_model.args.use_speaker_embedding}')
print(f'num_speakers: {vits_model.args.num_speakers}')

Trainable params: 1,792
use_speaker_embedding: True
num_speakers: 7


In [ ]:
import torch.nn as nn
from torch.nn.utils import weight_norm

def inject_cond_layers(vits_model, gin_channels=256):
    """Inject conditioning layers into modules that lack them."""
    wn_count, sdp_count = 0, 0
    for module in vits_model.modules():
        mtype = type(module).__name__

        # WaveNet modules
        if mtype == 'WN':
            if not hasattr(module, 'cond_layer') or module.cond_layer is None:
                hidden_channels = getattr(module, 'hidden_channels', 192)
                n_layers = getattr(module, 'n_layers', len(module.in_layers) if hasattr(module, 'in_layers') else 4)
                cond_layer = nn.Conv1d(gin_channels, 2 * hidden_channels * n_layers, 1)
                module.cond_layer = weight_norm(cond_layer, name='weight').to(DEVICE)
                module.gin_channels = gin_channels
                wn_count += 1

        # Stochastic duration predictor
        if mtype == 'StochasticDurationPredictor':
            if not hasattr(module, 'cond') or module.cond is None:
                # Find filter_channels from existing attributes
                filter_channels = getattr(module, 'filter_channels', 192)
                module.cond = nn.Conv1d(gin_channels, filter_channels, 1).to(DEVICE)
                sdp_count += 1

    print(f'Injected cond_layer into {wn_count} WN modules')
    print(f'Injected cond into {sdp_count} StochasticDurationPredictor modules')

inject_cond_layers(vits_model, gin_channels=EMB_DIM)

# Freeze all except emb_g and injected cond layers
for name, param in vits_model.named_parameters():
    param.requires_grad = ('emb_g' in name) or ('cond_layer' in name) or ('.cond.' in name) or name.endswith('.cond.weight') or name.endswith('.cond.bias')

trainable = sum(p.numel() for p in vits_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in vits_model.parameters())
print(f'Trainable: {trainable:,} / {total:,} params ({100*trainable/total:.3f}%)')

# Print which params are trainable for sanity check
print('\nTrainable param names:')
for name, param in vits_model.named_parameters():
    if param.requires_grad:
        print(f'  {name}: {param.shape}')

# Rebuild optimizer
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, vits_model.parameters()),
    lr=LR, betas=(0.8, 0.99), eps=1e-9
)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.999875)
print('\nOptimizer rebuilt.')

Injected cond_layer into 0 WN modules
Injected cond into 1 StochasticDurationPredictor modules
Trainable: 3,221,440 / 86,280,620 params (3.734%)

Trainable param names:
  posterior_encoder.enc.cond_layer.bias: torch.Size([6144])
  posterior_encoder.enc.cond_layer.weight_g: torch.Size([6144, 1, 1])
  posterior_encoder.enc.cond_layer.weight_v: torch.Size([6144, 256, 1])
  flow.flows.0.enc.cond_layer.bias: torch.Size([1536])
  flow.flows.0.enc.cond_layer.weight_g: torch.Size([1536, 1, 1])
  flow.flows.0.enc.cond_layer.weight_v: torch.Size([1536, 256, 1])
  flow.flows.1.enc.cond_layer.bias: torch.Size([1536])
  flow.flows.1.enc.cond_layer.weight_g: torch.Size([1536, 1, 1])
  flow.flows.1.enc.cond_layer.weight_v: torch.Size([1536, 256, 1])
  flow.flows.2.enc.cond_layer.bias: torch.Size([1536])
  flow.flows.2.enc.cond_layer.weight_g: torch.Size([1536, 1, 1])
  flow.flows.2.enc.cond_layer.weight_v: torch.Size([1536, 256, 1])
  flow.flows.3.enc.cond_layer.bias: torch.Size([1536])
  flow.flows.

In [ ]:
# Replace the save section at end of each epoch with:
cond_state = {k: v.cpu() for k, v in vits_model.state_dict().items()
              if 'emb_g' in k or 'cond_layer' in k or '.cond.' in k}

ckpt = {
    'epoch':       epoch,
    'cond_state':  cond_state,
    'optimizer':   optimizer.state_dict(),
    'scheduler':   scheduler.state_dict(),
    'loss':        avg_train,
    'val_loss':    avg_val,
}
torch.save(ckpt, CKPT_PATH)
torch.save(ckpt, EXP_DIR / f'{epoch}epoch.pth')
if avg_val < best_val_loss:
    best_val_loss = avg_val
    torch.save(ckpt, EXP_DIR / 'best.pth')
    print(f'  ★ New best val loss: {best_val_loss:.4f} → best.pth')

In [ ]:
start_epoch = 4
best_val_loss = 3.099  # val loss from epoch 3

In [ ]:
import torch
import torch.nn.functional as F
import time

def audio_to_spec(audio, n_fft=1024, hop_length=256, win_length=1024):
    window = torch.hann_window(win_length).to(audio.device)
    spec = torch.stft(
        audio, n_fft=n_fft, hop_length=hop_length, win_length=win_length,
        window=window, center=True, return_complex=True
    )
    return torch.abs(spec)

def mel_loss_fn(pred_wav, target_wav, sr=22050):
    import torchaudio
    mel_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=sr, n_fft=1024, hop_length=256,
        win_length=1024, n_mels=80
    ).to(pred_wav.device)
    pred_mel   = mel_transform(pred_wav.unsqueeze(1) if pred_wav.dim() == 1 else pred_wav)
    target_mel = mel_transform(target_wav.unsqueeze(1) if target_wav.dim() == 1 else target_wav)
    min_len = min(pred_mel.shape[-1], target_mel.shape[-1])
    return F.l1_loss(pred_mel[..., :min_len], target_mel[..., :min_len])

best_val_loss = float('inf')

for epoch in range(start_epoch, MAX_EPOCHS + 1):
    epoch_start  = time.time()
    train_losses = []
    vits_model.train()

    for batch_idx, (audio, lengths, emotions, texts) in enumerate(train_loader):
        audio    = audio.to(DEVICE)
        emotions = emotions.to(DEVICE)

        y_spec = audio_to_spec(audio)
        y_spec_lengths = (lengths.to(DEVICE) // 256) + 1

        tokenizer = tts_base.synthesizer.tts_model.tokenizer
        token_ids = [torch.LongTensor(tokenizer.text_to_ids(t)) for t in texts]
        max_text_len = max(t.shape[0] for t in token_ids)
        x = torch.zeros(len(token_ids), max_text_len, dtype=torch.long).to(DEVICE)
        x_lengths = torch.LongTensor([t.shape[0] for t in token_ids]).to(DEVICE)
        for i, t in enumerate(token_ids):
            x[i, :t.shape[0]] = t.to(DEVICE)

        optimizer.zero_grad()
        try:
            outputs = vits_model.forward(
                x=x,
                x_lengths=x_lengths,
                y=y_spec,
                y_lengths=y_spec_lengths,
                waveform=audio.unsqueeze(1),
                aux_input={'speaker_ids': emotions, 'd_vectors': None, 'language_ids': None}
            )

            pred_wav = None
            if isinstance(outputs, dict):
                pred_wav = outputs.get('model_outputs',
                           outputs.get('waveform_seg',
                           outputs.get('y_hat')))
            if pred_wav is None:
                continue
            pred_wav = pred_wav.squeeze(1) if pred_wav.dim() == 3 else pred_wav

            target = audio[:, :pred_wav.shape[-1]] if audio.shape[-1] >= pred_wav.shape[-1] else audio
            loss = mel_loss_fn(pred_wav, target)

            if torch.isnan(loss) or torch.isinf(loss):
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                [p for p in vits_model.parameters() if p.requires_grad], 5.0
            )
            optimizer.step()
            train_losses.append(loss.item())

            if (batch_idx + 1) % LOG_EVERY == 0:
                avg = sum(train_losses[-LOG_EVERY:]) / min(LOG_EVERY, len(train_losses))
                print(f'  Epoch {epoch} | Batch {batch_idx+1}/{len(train_loader)} | '
                      f'Loss: {avg:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}')
        except Exception as ex:
            if batch_idx < 3:
                import traceback
                traceback.print_exc()
            continue

    scheduler.step()

    # Validation
    vits_model.eval()
    val_losses = []
    with torch.no_grad():
        for audio, lengths, emotions, texts in val_loader:
            audio    = audio.to(DEVICE)
            emotions = emotions.to(DEVICE)
            y_spec = audio_to_spec(audio)
            y_spec_lengths = (lengths.to(DEVICE) // 256) + 1

            tokenizer = tts_base.synthesizer.tts_model.tokenizer
            token_ids = [torch.LongTensor(tokenizer.text_to_ids(t)) for t in texts]
            max_text_len = max(t.shape[0] for t in token_ids)
            x = torch.zeros(len(token_ids), max_text_len, dtype=torch.long).to(DEVICE)
            x_lengths = torch.LongTensor([t.shape[0] for t in token_ids]).to(DEVICE)
            for i, t in enumerate(token_ids):
                x[i, :t.shape[0]] = t.to(DEVICE)

            try:
                outputs = vits_model.forward(
                    x=x, x_lengths=x_lengths,
                    y=y_spec, y_lengths=y_spec_lengths,
                    waveform=audio.unsqueeze(1),
                    aux_input={'speaker_ids': emotions, 'd_vectors': None, 'language_ids': None}
                )
                pred_wav = outputs.get('model_outputs',
                           outputs.get('waveform_seg',
                           outputs.get('y_hat')))
                if pred_wav is None: continue
                pred_wav = pred_wav.squeeze(1) if pred_wav.dim() == 3 else pred_wav
                target = audio[:, :pred_wav.shape[-1]] if audio.shape[-1] >= pred_wav.shape[-1] else audio
                val_losses.append(mel_loss_fn(pred_wav, target).item())
            except Exception:
                continue

    avg_train = sum(train_losses) / len(train_losses) if train_losses else float('nan')
    avg_val   = sum(val_losses)   / len(val_losses)   if val_losses   else float('nan')
    elapsed   = time.time() - epoch_start
    print(f'Epoch {epoch}/{MAX_EPOCHS} | Train: {avg_train:.4f} | Val: {avg_val:.4f} | '
          f'Time: {elapsed:.0f}s | LR: {scheduler.get_last_lr()[0]:.2e}')

    # Save full conditioning state (emb_g + all cond_layers)
    cond_state = {k: v.cpu() for k, v in vits_model.state_dict().items()
                  if 'emb_g' in k or 'cond_layer' in k or '.cond.' in k}

    ckpt = {
        'epoch':       epoch,
        'cond_state':  cond_state,
        'optimizer':   optimizer.state_dict(),
        'scheduler':   scheduler.state_dict(),
        'loss':        avg_train,
        'val_loss':    avg_val,
    }
    torch.save(ckpt, CKPT_PATH)
    torch.save(ckpt, EXP_DIR / f'{epoch}epoch.pth')
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(ckpt, EXP_DIR / 'best.pth')
        print(f'  ★ New best val loss: {best_val_loss:.4f} → best.pth')

print('Training complete!')

  Epoch 4 | Batch 50/1165 | Loss: 3.2406 | LR: 2.00e-04
  Epoch 4 | Batch 100/1165 | Loss: 3.1029 | LR: 2.00e-04
  Epoch 4 | Batch 150/1165 | Loss: 3.1341 | LR: 2.00e-04
  Epoch 4 | Batch 200/1165 | Loss: 3.1294 | LR: 2.00e-04
  Epoch 4 | Batch 250/1165 | Loss: 3.0488 | LR: 2.00e-04
  Epoch 4 | Batch 300/1165 | Loss: 3.1687 | LR: 2.00e-04
  Epoch 4 | Batch 350/1165 | Loss: 3.1592 | LR: 2.00e-04
  Epoch 4 | Batch 400/1165 | Loss: 3.1723 | LR: 2.00e-04
  Epoch 4 | Batch 450/1165 | Loss: 3.1893 | LR: 2.00e-04
  Epoch 4 | Batch 500/1165 | Loss: 3.2374 | LR: 2.00e-04
  Epoch 4 | Batch 550/1165 | Loss: 3.0321 | LR: 2.00e-04
  Epoch 4 | Batch 600/1165 | Loss: 3.0987 | LR: 2.00e-04
  Epoch 4 | Batch 650/1165 | Loss: 3.0756 | LR: 2.00e-04
  Epoch 4 | Batch 700/1165 | Loss: 3.3443 | LR: 2.00e-04
  Epoch 4 | Batch 750/1165 | Loss: 3.2561 | LR: 2.00e-04
  Epoch 4 | Batch 800/1165 | Loss: 3.2760 | LR: 2.00e-04
  Epoch 4 | Batch 850/1165 | Loss: 3.0671 | LR: 2.00e-04
  Epoch 4 | Batch 900/1165 | Los

KeyboardInterrupt: 

## 16) Load Finetuned Model

In [ ]:
import torch

best_ckpt = EXP_DIR / 'best.pth'

print(f'Loading: {best_ckpt}')
ckpt = torch.load(best_ckpt, map_location=DEVICE, weights_only=False)

if 'cond_state' in ckpt:
    missing, unexpected = vits_model.load_state_dict(ckpt['cond_state'], strict=False)
    print(f'Loaded cond_state: {len(ckpt["cond_state"])} tensors')
else:
    vits_model.emb_g.weight.data = ckpt['emb_g_weight'].to(DEVICE)
    print('Loaded emb_g only (cond_layers will be random)')

vits_model.eval()
print(f'Epoch: {ckpt.get("epoch", "?")} | val_loss: {ckpt.get("val_loss", "?")}')

def emotion_tts(text, emotion_label, noise_scale=0.333):
    sid = torch.LongTensor([EMOTION_TO_ID[emotion_label.lower()]]).to(DEVICE)
    tokenizer = tts_base.synthesizer.tts_model.tokenizer
    ids = tokenizer.text_to_ids(text)
    x = torch.LongTensor(ids).unsqueeze(0).to(DEVICE)
    x_lengths = torch.LongTensor([len(ids)]).to(DEVICE)
    with torch.no_grad():
        outputs = vits_model.inference(
            x,
            aux_input={
                'x_lengths':    x_lengths,
                'speaker_ids':  sid,
                'd_vectors':    None,
                'language_ids': None,
                'durations':    None,
            }
        )
    wav = outputs.get('model_outputs', outputs.get('waveform', outputs))
    wav_np = wav.squeeze().cpu().numpy()
    wav_np = wav_np / (abs(wav_np).max() + 1e-8)
    return wav_np, SR

print('emotion_tts() ready.')

Loading: /content/drive/MyDrive/vits_meld_coqui_st/best.pth
Loaded cond_state: 18 tensors
Epoch: 10 | val_loss: 3.0943167666866356
emotion_tts() ready.


## 17) Listen to All 7 Emotions — After Finetuning

In [ ]:
import soundfile as sf
from IPython.display import Audio, display
import time

synthesis_text = 'Ich kann nicht glauben, dass das gerade passiert.'
print(f'Text: {synthesis_text}\n')

for emo in EMOTIONS:
    start = time.time()
    wav_np, sr = emotion_tts(synthesis_text, emo)
    rtf = (time.time() - start) / (len(wav_np) / sr)
    sf.write(f'tts_result/coqui_st_{emo}.wav', wav_np, sr)
    print(f'{emo:12s} (ID={EMOTION_TO_ID[emo]})  RTF={rtf:.3f}')
    display(Audio(wav_np, rate=sr))
    print()

Text: Ich kann nicht glauben, dass das gerade passiert.

neutral      (ID=0)  RTF=0.054



surprise     (ID=1)  RTF=0.049



fear         (ID=2)  RTF=0.055



sadness      (ID=3)  RTF=0.054



joy          (ID=4)  RTF=0.044



disgust      (ID=5)  RTF=0.044



anger        (ID=6)  RTF=0.055


## 18) F0 Analysis — Baseline vs Finetuned Comparison

In [ ]:
import librosa, numpy as np, matplotlib.pyplot as plt

synthesis_text = 'Ich kann nicht glauben, dass das gerade passiert.'
f0_finetuned = {}

for emo in EMOTIONS:
    wav_np, sr = emotion_tts(synthesis_text, emo)
    f0 = librosa.yin(wav_np, fmin=60, fmax=400, sr=sr)
    f0_voiced = f0[(f0 > 60) & (f0 < 400)]
    f0_finetuned[emo] = float(np.mean(f0_voiced)) if len(f0_voiced) > 0 else 0.0
    print(f'{emo:12s}: {f0_finetuned[emo]:.1f} Hz')

# Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
colors = ['gray', 'purple', 'blue', 'steelblue', 'gold', 'green', 'red']

# Baseline
axes[0].bar(EMOTIONS, [baseline_f0] * len(EMOTIONS), color=colors)
axes[0].set_title('Baseline (no emotion conditioning)', fontsize=12)
axes[0].set_ylabel('Mean F0 (Hz)')
axes[0].set_ylim(0, max(f0_finetuned.values()) * 1.3 + 20)
axes[0].tick_params(axis='x', rotation=15)

# Finetuned
axes[1].bar(f0_finetuned.keys(), f0_finetuned.values(), color=colors)
axes[1].set_title('After Finetuning (with emotion conditioning)', fontsize=12)
axes[1].set_ylabel('Mean F0 (Hz)')
axes[1].set_ylim(0, max(f0_finetuned.values()) * 1.3 + 20)
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle('F0 Comparison: Baseline vs Emotion-Conditioned VITS', fontsize=13)
plt.tight_layout()
plt.savefig('tts_result/f0_comparison.png', dpi=100)
plt.show()
print('Saved: tts_result/f0_comparison.png')

neutral     : 143.3 Hz
surprise    : 138.1 Hz
fear        : 138.4 Hz
sadness     : 142.3 Hz
joy         : 142.4 Hz
disgust     : 134.3 Hz
anger       : 149.6 Hz
Saved: tts_result/f0_comparison.png


## 19) Mel Spectrogram Comparison

In [ ]:
import librosa, librosa.display, numpy as np, matplotlib.pyplot as plt

synthesis_text = 'Ich kann nicht glauben, dass das gerade passiert.'
fig, axes = plt.subplots(NUM_CLASSES, 1, figsize=(14, 3 * NUM_CLASSES))

for i, emo in enumerate(EMOTIONS):
    wav_np, sr = emotion_tts(synthesis_text, emo)
    mel     = librosa.feature.melspectrogram(y=wav_np, sr=sr, n_fft=1024, hop_length=256, n_mels=80)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    librosa.display.specshow(log_mel, sr=sr, hop_length=256,
                             x_axis='time', y_axis='mel', ax=axes[i])
    axes[i].set_title(f'{emo}  (ID={EMOTION_TO_ID[emo]})', fontsize=12)
    axes[i].label_outer()

plt.suptitle('Same text, 7 emotion IDs — Fine-tuned Coqui VITS (MELD-ST)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('tts_result/emotion_spectrograms.png', bbox_inches='tight', dpi=100)
plt.show()
print('Saved: tts_result/emotion_spectrograms.png')

Saved: tts_result/emotion_spectrograms.png


## 20) Pipeline Integration Function

In [ ]:
def emotion_tts_pipeline(translated_german_text: str, emotion_label: str, device: str = 'cuda'):
    """
    Drop-in TTS function for the full pipeline.

    Args:
        translated_german_text: German text from translation module
        emotion_label:          From emotion recognizer.
                                One of: neutral, surprise, fear, sadness, joy, disgust, anger
        device:                 'cuda' or 'cpu'

    Returns:
        (np.ndarray waveform float32, int sample_rate)
    """
    return emotion_tts(translated_german_text, emotion_label)


# Demo with different sentences
from IPython.display import Audio, display

test_cases = [
    ('Wir müssen über das Geschehene reden.', 'anger'),
    ('Das ist wirklich unglaublich schön!', 'joy'),
    ('Ich vermisse dich so sehr.', 'sadness'),
]

for text, emo in test_cases:
    wav_np, sr = emotion_tts_pipeline(text, emo)
    print(f'{emo} | {text}')
    display(Audio(wav_np, rate=sr))
    print()